In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
import matplotlib.pyplot as plt

u = mda.Universe("rfc.psf", "rfc.dcd")

# --- Select the C-alpha atoms of segment PROE ---
# We often use C-alphas because they represent the "trace" of the protein backbone
PROE = u.select_atoms("segid PROE and name CA")

# --- Explicitly align our trajectory on PROE ---
aligner = align.AlignTraj(u, u, select="segid PROE and name CA", in_memory=True)
aligner.run()

# --- Run RMSF ---
# Note: You should align the trajectory to a reference structure first 
# to ensure you aren't measuring global rotation as "fluctuation"
rms_analysis = rms.RMSF(PROE).run()

# --- Plotting ---
plt.plot(PROE.resids, rms_analysis.results.rmsf)
plt.xlabel("Residue ID")
plt.ylabel("RMSF (Å)")
plt.title("RMSF of Segment PROE (MDAnalysis)")
plt.show()

In [ ]:
import mdtraj as md

traj = md.load("rfc.dcd", top="rfc.psf")

# --- Select C-alphas in segment PROE ---
selection = traj.topology.select("chainid 4 and name CA")

# --- Calculate RMSF ---
# again we need to align our trajectory explicitly for RMSF
traj_aligned = traj.superpose(traj, 0, atom_indices=selection) 
md_rmsf = md.rmsf(traj_aligned, traj_aligned[0], atom_indices=selection)

# --- Plotting ---
# We get the residue numbers for the x-axis
res_numbers = [traj.topology.atom(i).residue.resSeq for i in selection]

plt.plot(res_numbers, md_rmsf, color='orange')
plt.xlabel("Residue ID")
plt.ylabel("RMSF (nm)")
plt.title("RMSF of Segment PROE (MDTraj)")
plt.show()

### **Homework for the reader**
Write a script that takes the RMSF values from one of the above analysis and writes it as the B-factor in a PDB file so that you can visualize with ChimeraX or PyMol.